# Draw SegFormer v1 — treino sintético privado

Treino single-frame 4 classes com máscaras sintéticas pixel a pixel. Esta execução é **exploratória**: não usa nem valida qualidade em câmera real. O pool de revisão, os pseudo-rótulos reais e dados temporais não são inputs.

O split de teste é avaliado uma única vez depois da seleção do checkpoint pela validação. Não há publicação/atualização de datasets.


## Ambiente e gate de GPU/ONNX

Se ONNX/ONNX Runtime não vierem no ambiente Kaggle, esta célula instala versões fixas. Qualquer ausência ou falha interrompe a execução. O runtime precisa oferecer CUDA; não inicia treino em CPU.


In [ ]:
import importlib, subprocess, sys, torch

for package, module in (("onnx==1.17.0", "onnx"), ("onnxruntime==1.20.1", "onnxruntime")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", package], check=True)
        importlib.invalidate_caches()
        importlib.import_module(module)

for name in ("torch", "transformers", "cv2", "numpy", "onnx", "onnxruntime"):
    mod = importlib.import_module(name)
    print(name, getattr(mod, "__version__", "OK"))
if not torch.cuda.is_available():
    raise RuntimeError("GPU CUDA ausente; nenhum treino iniciado")
print("GPU", torch.cuda.get_device_name(0))

# Use apenas o cache privado anexado; não baixe pesos silenciosamente.
import glob, os
cache_roots = [p for p in glob.glob("/kaggle/input/**/hub", recursive=True) if os.path.isdir(os.path.join(p, "models--nvidia--mit-b0"))]
if not cache_roots:
    raise RuntimeError("Cache privado nvidia/mit-b0 ausente")
os.environ["HF_HUB_CACHE"] = cache_roots[0]
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print("cache de pesos", cache_roots[0])


## Código e configuração congelados

O código abaixo foi empacotado dentro deste notebook para que a execução use os arquivos atuais do branch sem criar uma nova versão de dataset Kaggle. Os hashes são registrados antes do treino.


In [ ]:
import base64, hashlib
from pathlib import Path

BUNDLE = Path("/kaggle/working/draw/training")
BUNDLE.mkdir(parents=True, exist_ok=True)
TRAIN_B64 = "..."
CONFIG_B64 = "ewogICJkYXRhIjogewogICAgImF1Z21lbnRhdGlvbiI6IHsKICAgICAgImdlb21ldHJpYyI6IFsKICAgICAgICAiaG9yaXpvbnRhbF9mbGlwX3A9MC41IiwKICAgICAgICAicm90YXRpb25fZGVncmVlcz0xNSIsCiAgICAgICAgInNjYWxlPTAuOS0xLjEiLAogICAgICAgICJwZXJzcGVjdGl2ZV9saWdodCIKICAgICAgXSwKICAgICAgIm5vdGUiOiAiQXVnbWVudGF0aW9uIGdlb23DqXRyaWNhIGFwbGljYS1zZSBJREVOVElDQU1FTlRFIMOgIGltYWdlbSBlIMOgIG3DoXNjYXJhIChudW5jYSBzw7Mgw6AgaW1hZ2VtKS4gQXVnbWVudGF0aW9uIGZvdG9tw6l0cmljYSBzw7Mgw6AgaW1hZ2VtLiBNw6FzY2FyYSBudW5jYSDDqSBib3JyYWRhIG5lbSBpbnRlcnBvbGFkYSBjb20gYmlsaW5lYXI6IG5lYXJlc3QtbmVpZ2hib3IsIHBhcmEgbsOjbyBjcmlhciBjbGFzc2VzIGZyYWNpb27DoXJpYXMuIiwKICAgICAgInBob3RvbWV0cmljIjogWwogICAgICAgICJicmlnaHRuZXNzX2NvbnRyYXN0PTAuMiIsCiAgICAgICAgImdhdXNzaWFuX2JsdXJfcD0wLjIiLAogICAgICAgICJnYXVzc2lhbl9ub2lzZV9wPTAuMiIsCiAgICAgICAgImh1ZV9zYXR1cmF0aW9uX3ZhbHVlX2xpZ2h0IgogICAgICBdCiAgICB9LAogICAgImZyZWloYW5kIjogewogICAgICAia2FnZ2xlX2RhdGFzZXQiOiAiZGFuaWVsZGVscm8vZnJlaWhhbmQiLAogICAgICAibm90ZSI6ICJPIGdlcmFkb3Igc2ludMOpdGljbyDDqSBkZSBvdXRybyBhZ2VudGU7IGVzdGUgdHJlaW5hbWVudG8gYXBlbmFzIGNvbnNvbWUgbyBkYXRhc2V0IGdlcmFkby4gTsOjbyBiYWl4YXIgbyBGcmVpSEFORCBpbnRlaXJvIG5lc3RhIG3DoXF1aW5hOiBhIGNoZWNhZ2VtIGRlIGZvcm1hdG8gcm9kYSBubyBLYWdnbGUsIG9uZGUgbyBkYXRhc2V0IGVudHJhIGNvbW8gaW5wdXQuIiwKICAgICAgInVzZSI6ICJTb21lbnRlIGNvbW8gbWF0w6lyaWEtcHJpbWEgZGFzIG3Do29zIG5vIGdlcmFkb3IgKGltYWdlbSBSR0IgKyBtw6FzY2FyYSBkYSBtw6NvKSBlIGNvbW8gY2hlY2FnZW0gZGUgZm9ybWF0byBubyBub3RlYm9vay4gS2V5cG9pbnRzLCBwb3NlIDNELCBtZXNoIGUgY8OibWVyYSBuw6NvIHPDo28gdXNhZG9zLCBzYWx2byBiZW5lZsOtY2lvIGRpcmV0by4gU2UgYSB2ZXJzw6NvIGRvIEthZ2dsZSBuw6NvIHRpdmVyIG3DoXNjYXJhcyBkZSBtw6NvIHV0aWxpesOhdmVpcywgTsODTyBpbnZlbnRhciByw7N0dWxvczogZG9jdW1lbnRhciBlIHRyb2NhciBhIGZvbnRlLCByZWdpc3RyYW5kbyBsaWNlbsOnYSBlIG9yaWdlbS4iCiAgICB9LAogICAgImxheW91dCI6IHsKICAgICAgIm5vdGUiOiAiT3Mgc3BsaXRzIErDgSB2w6ptIHNlcGFyYWRvcyBwZWxvIGdlcmFkb3IsIHBvciBmb250ZSBvcmlnaW5hbCAobcOjbywgZm9saGEsIGZ1bmRvKSwgbnVuY2EgcG9yIHNwbGl0IGFsZWF0w7NyaW8gZGVwb2lzIGRlIGdlcmFyLiB0cmFpbi5weSBuw6NvIHJlZW1iYXJhbGhhIGZvbnRlcyBlbnRyZSBzcGxpdHM7IGVsZSBhcGVuYXMgbMOqIGFzIHBhc3RhcyBlIHZhbGlkYSBvIHNwbGl0X21hbmlmZXN0Lmpzb24sIHNlIHByZXNlbnRlLiIsCiAgICAgICJwZXJfc3BsaXQiOiBbCiAgICAgICAgImltYWdlcy8qLnBuZyAob3UgLmpwZykiLAogICAgICAgICJtYXNrcy8qLnBuZyAobWVzbW8gbm9tZSBiYXNlKSIsCiAgICAgICAgImlua19mdWxsLyoucG5nIChvcGNpb25hbCkiCiAgICAgIF0sCiAgICAgICJzcGxpdHMiOiBbCiAgICAgICAgInRyYWluIiwKICAgICAgICAidmFsIiwKICAgICAgICAidGVzdCIKICAgICAgXSwKICAgICAgInN5bnRoZXRpY19yb290IjogImRhdGFzZXQvc3ludGhldGljIChvdSAva2FnZ2xlL2lucHV0LzxkYXRhc2V0Pi9kYXRhc2V0L3N5bnRoZXRpYykiCiAgICB9LAogICAgInJlYWxfaG9sZG91dCI6IHsKICAgICAgIm5vdGUiOiAiTm8gaW5kZXBlbmRlbnQgcmVhbCBob2xkb3V0IGlzIGF2YWlsYWJsZSBmcm9tIHRoZSB0d28gY2FwdHVyZWQgc2Vzc2lvbnMgYWZ0ZXIgdGhleSBhcmUgYXNzaWduZWQgdG8gcmVhbF90cmFpbi4gUmVzZXJ2ZSBhIG5ldyBjYXB0dXJlIHNlc3Npb24gZm9yIHJlYWwgaG9sZG91dCBiZWZvcmUgY2xhaW1pbmcgcmVhbC13b3JsZCBnZW5lcmFsaXphdGlvbi4iCiAgICB9LAogICAgInNwbGl0X3JhdGlvIjogewogICAgICAidGVzdCI6IDAuMSwKICAgICAgInRyYWluIjogMC44LAogICAgICAidmFsIjogMC4xCiAgICB9LAogICAgInNwbGl0X3NlZWQiOiA0MiwKICAgICJtYXNrX2VuY29kaW5nIjogewogICAgICAiY2xhc3NlcyI6IHsKICAgICAgICAiMCI6ICJiYWNrZ3JvdW5kIiwKICAgICAgICAiMSI6ICJ2aXNpYmxlIHBhcGVyIiwKICAgICAgICAiMiI6ICJ2aXNpYmxlIGhhbmQiLAogICAgICAgICIzIjogInVzZXItYWRkZWQgdmlzaWJsZSBpbmsiCiAgICAgIH0sCiAgICAgICJpZ25vcmVfaW5kZXgiOiAyNTUsCiAgICAgICJpZ25vcmVfbm90ZSI6ICIyNTUgbWFya3MgcGl4ZWxzIGRlbGliZXJhdGVseSBsZWZ0IHVua25vd247IHRyYWluLnB5IGV4Y2x1ZGVzIHRoZW0gZnJvbSBjcm9zcy1lbnRyb3B5LCBEaWNlLCBwaXhlbCBjb3VudHMsIGFuZCBldmFsdWF0aW9uLiBOZXZlciBjb252ZXJ0IDI1NSB0byBiYWNrZ3JvdW5kLiIsCiAgICAgICJ2aXNpYmxlX3BpeGVsX3J1bGUiOiAiQSBoYW5kIGNvdmVyaW5nIGluayBpcyBjbGFzcyAyOyBwcmludGVkIHRleHQgcmVtYWlucyBjbGFzcyAxLiIKICAgIH0sCiAgICAicmVhbF90cmFpbiI6IHsKICAgICAgInNvdXJjZV9tYW5pZmVzdCI6ICJjYXB0dXJlcy90cmlhZ2UvY2xhc3NpZmllZF9hbGxfY2FuZGlkYXRlcy5jc3YiLAogICAgICAicmV0YWluZWRfZnJhbWVzIjogNzE5LAogICAgICAic2Vzc2lvbnMiOiBbCiAgICAgICAgInNlc3Npb25fMjAyNjEwMjlfMTMwMDEzXzQ1MzQxOSIsCiAgICAgICAgInNlc3Npb25fMjAyNjEwMjlfMTUxMTI2XzA4OTU1NyIKICAgICAgXSwKICAgICAgInNwbGl0IjogInRyYWluIiwKICAgICAgIm5vdGUiOiAiUGVyIHVzZXIgaW5zdHJ1Y3Rpb24sIGJvdGggY29tcGxldGUgc291cmNlIHNlc3Npb25zIGFyZSB3aG9sbHkgYXNzaWduZWQgdG8gcmVhbCB0cmFpbi4gVGhlIGxvY2FsIHBzZXVkby1sYWJlbCBwYWNrYWdlIG11c3QgYmUgcGFzc2VkIHdpdGggLS1yZWFsLXRyYWluLWRpcjsgbm90ZWJvb2sgYWJvcnRzIGlmIHRoZSBwcml2YXRlIGlucHV0IGlzIG1pc3NpbmcuIFN5bnRoZXRpYyB2YWwvdGVzdCBvbmx5OyBubyBpbmRlcGVuZGVudCByZWFsIGhvbGRvdXQuIFRyZWF0IHJlc3VsdHMgYXMgZXhwbG9yYXRvcnkgdW50aWwgbGFiZWxzIGFyZSByZXZpZXdlZC4iLAogICAgICAibGFiZWxfc3RhdHVzIjogInBzZXVkby1sYWJlbCBwcm9wb3NhbHMsIG5vdCBodW1hbi1yZXZpZXdlZCBncm91bmQgdHJ1dGgiCiAgICB9CiAgfSwKICAiZGVzY3JpcHRpb24iOiAiU2VnRm9ybWVyLUIwLCA0IGNsYXNzZXMgKGJhY2tncm91bmQvcGFwZXIvaGFuZC9pbmspLCBwYXJhIG8gRHJhdy4gVHJlaW5vIG5vIEthZ2dsZSBjb20gR1BVLiBFc3RlIGFycXVpdm8gw6kgYSDDum5pY2EgZm9udGUgZGUgaGlwZXJwYXLDom1ldHJvcyBkbyB0cmVpbmFtZW50bzsgbyBub3RlYm9vayBhcGVuYXMgbyBsw6ouIiwKICAiZXhwb3J0IjogewogICAgImFydGlmYWN0cyI6IFsKICAgICAgImRyYXdfc2VnbWVudGF0aW9uX3YxLm9ubngiLAogICAgICAiY2xhc3Nlcy5qc29uIiwKICAgICAgIm1ldHJpY3MuanNvbiIsCiAgICAgICJ0cmFpbmluZ19jb25maWcuanNvbiIsCiAgICAgICJzcGxpdF9tYW5pZmVzdC5qc29uIgogICAgXSwKICAgICJkeW5hbWljX2F4ZXMiOiB7CiAgICAgICJpbWFnZSI6IFsKICAgICAgICAwCiAgICAgIF0sCiAgICAgICJsb2dpdHMiOiBbCiAgICAgICAgMAogICAgICBdCiAgICB9LAogICAgImlucHV0X25hbWUiOiAiaW1hZ2UiLAogICAgIm9ubngiOiAiZHJhd19zZWdtZW50YXRpb25fdjEub25ueCIsCiAgICAib3BzZXRfdmVyc2lvbiI6IDE0LAogICAgIm1heF9hYnNfZGlmZl90b2xlcmFuY2UiOiAwLjAwMSwKICAgICJtaW5fYXJnbWF4X2lvdSI6IDAuOTksCiAgICAib3V0cHV0X25hbWUiOiAibG9naXRzIiwKICAgICJ2YWxpZGF0aW9uIjogIlJvZGFyIG8gbW9kZWxvIFB5VG9yY2ggZSBvIE9OTlggbm8gbWVzbW8gbG90ZSBkZSB2YWxpZGHDp8OjbyBlIGNvbXBhcmFyOiBkaWZlcmVuw6dhIG3DoXhpbWEgYWJzb2x1dGEgZG9zIGxvZ2l0cyBlIElvVSBwb3IgY2xhc3NlLiBEZWdyYWRhw6fDo28gc2lnbmlmaWNhdGl2YSBhYm9ydGEgYSBlbnRyZWdhLiBPIHJlc3VsdGFkbyB2YWkgcGFyYSBtZXRyaWNzLmpzb24gKGNhbXBvIG9ubnhfdmFsaWRhdGlvbikuIgogIH0sCiAgImthZ2dsZSI6IHsKICAgICJhY2NlbGVyYXRvciI6ICJHUFUgZGlzcG9uw612ZWwgbm8gS2FnZ2xlOyBjb25maXJtYXIgbyBhY2VsZXJhZG9yIG5vIHJ1bnRpbWUgYW50ZXMgZG8gdHJlaW5vLiIsCiAgICAiZXN0aW1hdGVkX3RpbWUiOiAiMzAgw6lwb2NhcywgZGF0YXNldCBzaW50w6l0aWNvIGRlIH41LTEwIG1pbCBpbWFnZW5zIDUxMng1MTIsIGJhdGNoIDg6IGVzdGltYXRpdmEgMy02IGhvcmFzIGVtIFQ0LiBEYXRhc2V0IG1lbm9yICgyIG1pbCBpbWFnZW5zKSBvdSAxNSDDqXBvY2FzOiAxLTIgaG9yYXMuIE8gdGVtcG8gcmVhbCBkZXBlbmRlIGRvIHRhbWFuaG8gZG8gZGF0YXNldCBnZXJhZG87IG8gbm90ZWJvb2sgcmVnaXN0cmEgbyB0ZW1wbyBwb3Igw6lwb2NhLiIsCiAgICAibm90ZSI6ICJUcmVpbm8gY29tcGxldG8gZGVwZW5kZSBkbyBkYXRhc2V0IHNpbnTDqXRpY28gZSBkbyBwYWNvdGUgcmVhbCBwc2V1ZG9fdHJhaW5pbmdfY2FuZGlkYXRlIGFuZXhhZG9zIGNvbW8gaW5wdXRzIHByaXZhZG9zLiBDb25maXJtZSBvIGFjZWxlcmFkb3IgcmVhbCBubyBydW50aW1lLiBPIHBhY290ZSByZWFsIGNvbnTDqW0gcHJvcG9zdGFzIGF1dG9tw6F0aWNhcyBuw6NvIHZlcmlmaWNhZGFzIGUgc2VydmUgYXBlbmFzIHBhcmEgdHJlaW5vIGV4cGxvcmF0w7NyaW8gYXTDqSByZXZpc8OjbyBodW1hbmEuIgogIH0sCiAgImxhYmVscyI6IHsKICAgICJpbmtfZnVsbF9tYXNrIjogIk3DoXNjYXJhIG9wY2lvbmFsLCBmb3JhIGRvIHRyZWlubywgY29tIG8gZGVzZW5obyBjb21wbGV0byBhbnRlcyBkYSBvY2x1c8OjbyAoYXJxdWl2byAqX2lua19mdWxsLnBuZykuIFNlcnZlIHPDsyBwYXJhIGF2YWxpYXIgYSBtZW3Ds3JpYSB0ZW1wb3JhbCBmdXR1cmEgZG8gRHJhdy4gTnVuY2EgZW50cmEgY29tbyBjbGFzc2UgbmVtIGNvbW8gZW50cmFkYS4iLAogICAgIm1hc2tfZW5jb2RpbmciOiAiU2luZ2xlLWNoYW5uZWwgUE5HIHdpdGggdmFsdWVzIDAsIDEsIDIsIDMsIG9yIDI1NSAodW5rbm93bjsgaWdub3JlZCBieSBsb3NzIGFuZCBtZXRyaWNzKS4iLAogICAgInJ1bGUiOiAiT25seSB2aXNpYmxlIHBpeGVscyBhcmUgbGFiZWxlZC4gSGFuZCBvdmVyIGluayBpcyBjbGFzcyAyOyBwcmludGVkIHRleHQgaXMgY2xhc3MgMS4gMjU1IGlzIG5ldmVyIGNvbnZlcnRlZCB0byBiYWNrZ3JvdW5kLiIKICB9LAogICJsb3NzIjogewogICAgImNsYXNzX3dlaWdodHNfcmF0aW9uYWxlIjogIkNsYXNzZXMgbXVpdG8gZGVzYmFsYW5jZWFkYXM6IGZ1bmRvIGRvbWluYSwgcGFwZWwgw6kgZ3JhbmRlLCBtw6NvIMOpIG3DqWRpYSwgaW5rIMOpIG1pbm9yaWEgKHRyYcOnbyBmaW5vID0gcG91Y29zIHBpeGVscykuIFBlc28gw6kgaW52ZXJzbyBhcHJveGltYWRvIGRhIGZyZXF1w6puY2lhLCBjb20gdGV0bywgcGFyYSBvIGluayBuw6NvIHNlciBpZ25vcmFkbyBzZW0gZG9taW5hciBvIGdyYWRpZW50ZS4gVmFsb3JlcyBpbmljaWFpcyBhIHBhcnRpciBkYSBjb250YWdlbSBkZSBwaXhlbHMgZG8gZGF0YXNldCBnZXJhZG87IG8gc2NyaXB0IHJlZ2lzdHJhIGEgY29udGFnZW0gcmVhbCBlLCBzZSBlbGEgZGl2ZXJnaXIgbXVpdG8sIG8gcGVzbyBkZXZlIHNlciByZWNhbGlicmFkbyBlIGEgbXVkYW7Dp2EgcmVnaXN0cmFkYSBlbSB0cmFpbmluZ19jb25maWcuanNvbiDigJQgbnVuY2EgYWp1c3RhciBwZXNvIG9saGFuZG8gbyBzcGxpdCBkZSB0ZXN0ZS4iLAogICAgImNyb3NzX2VudHJvcHlfY2xhc3Nfd2VpZ2h0cyI6IFsKICAgICAgMS4wLAogICAgICAyLjAsCiAgICAgIDMuMCwKICAgICAgMTAuMAogICAgXSwKICAgICJkaWNlIjogIkRpY2UgbXVsdGljbGFzc2UgKG3DqWRpYSBzb2JyZSBhcyA0IGNsYXNzZXMsIGNvbSBzbW9vdGg9MWUtNikuIERpY2UgY29tcGxlbWVudGEgbyBDRSBwb3JxdWUgbWVkZSBzb2JyZXBvc2nDp8OjbyBwb3IgY2xhc3NlLCBuw6NvIHPDsyBhY2VydG8gcG9yIHBpeGVsLCBlIMOpIG1lbm9zIHNlbnPDrXZlbCBhbyBkb23DrW5pbyBkbyBmdW5kby4iLAogICAgImZvcm11bGEiOiAibG9zcyA9IDAuNSAqIENyb3NzRW50cm9weV9jb21fcGVzbyArIDAuNSAqIERpY2VMb3NzX211bHRpY2xhc3NlIiwKICAgICJub3RlIjogIkFjY3VyYWN5IGdsb2JhbCBOw4NPIMOpIGNyaXTDqXJpbyBkZSBwYXJhZGEgbmVtIGRlIGNvbXBhcmHDp8OjbzogY29tIGZ1bmRvIGRvbWluYW50ZSwgdW0gbW9kZWxvIHF1ZSBzw7MgcHJldsOqIGZ1bmRvIGrDoSB0ZXJpYSBhY2N1cmFjeSBhbHRhIGUgc2VyaWEgaW7DunRpbC4iCiAgfSwKICAibWV0cmljcyI6IHsKICAgICJmaXhlZF9zY2VuYXJpb3MiOiBbCiAgICAgICJmb2xoYSBsaW1wYSIsCiAgICAgICJmb2xoYSBjb20gZGVzZW5obyIsCiAgICAgICJmb2xoYSBtdWl0byBkZXNlbmhhZGEiLAogICAgICAibcOjbyBjb2JyaW5kbyBjYW50byIsCiAgICAgICJtw6NvIGNvYnJpbmRvIGNlbnRybyIsCiAgICAgICJtw6NvIGNvYnJpbmRvIHRyYcOnbyIsCiAgICAgICJtw6NvIHBhcmNpYWxtZW50ZSBmb3JhIGRhIGltYWdlbSIsCiAgICAgICJmdW5kbyBjbGFybyIsCiAgICAgICJmdW5kbyBlc2N1cm8iLAogICAgICAic29tYnJhIGZvcnRlIiwKICAgICAgImZvbGhhIHBhcmNpYWxtZW50ZSBmb3JhIGRhIGltYWdlbSIsCiAgICAgICJtw6NvIHNlbSBmb2xoYSIsCiAgICAgICJvYmpldG8gYnJhbmNvIHNlbSBmb2xoYSIsCiAgICAgICJ0cmHDp28gZmlubyBkZSBsw6FwaXMiLAogICAgICAidHJhw6dvIGdyb3NzbyBkZSBjYW5ldGEiCiAgICBdLAogICAgImlua190b2xlcmFuY2UiOiB7CiAgICAgICJtZXRob2QiOiAiRGlsYXRhciBwcmVkacOnw6NvIGUgZ3JvdW5kIHRydXRoIGRlIGluayBwb3IgMi0zIHB4IChkaXNjbywgdmlhIGN2Mi5kaWxhdGUpIGUgcmVjYWxjdWxhciBwcmVjaXNpb24vcmVjYWxsIHNvYnJlIGEgdmVyc8OjbyBkaWxhdGFkYS4gVHJhw6dvIGZpbm8gZGVzbG9jYWRvIDEtMiBweCDDqSB2aXN1YWxtZW50ZSBjb3JyZXRvIG1hcyBkZXN0csOzaSBhIElvVSBlc3RyaXRhOyBhIG3DqXRyaWNhIGNvbSB0b2xlcsOibmNpYSBtZWRlIGlzc28gc2VtIGVzY29uZGVyIGEgSW9VIGVzdHJpdGEsIHF1ZSBjb250aW51YSBzZW5kbyByZXBvcnRhZGEgYW8gbGFkby4iLAogICAgICAibm90ZSI6ICJBbHRlcm5hdGl2YXMgZXF1aXZhbGVudGVzIChkaXN0YW5jZSB0cmFuc2Zvcm0sIGNvbXBhcmHDp8OjbyBwb3IgZXNxdWVsZXRvKSBzw6NvIGFjZWl0w6F2ZWlzIHNlIGRvY3VtZW50YWRhcywgbWFzIGEgZGlsYXRhw6fDo28gw6kgYSBpbXBsZW1lbnRhw6fDo28gZGUgcmVmZXLDqm5jaWEgZGVzdGUgcHJvamV0byBwb3Igc2VyIHNpbXBsZXMgZSByZXByb2R1esOtdmVsLiIsCiAgICAgICJyYWRpaV9weCI6IFsKICAgICAgICAyLAogICAgICAgIDMKICAgICAgXQogICAgfSwKICAgICJub3RlIjogIk9zIDE1IGNlbsOhcmlvcyBmaXhvcyB0w6ptIHByZWRpw6fDtWVzIHNhbHZhcyAoaW1hZ2VtICsgb3ZlcmxheSkgcGFyYSBjb21wYXJhw6fDo28gZW50cmUgdmVyc8O1ZXMgZG8gbW9kZWxvLiBNw6l0cmljYXMgc2ludMOpdGljYXMgZSBkbyBob2xkb3V0IHJlYWwgbnVuY2Egc8OjbyBtaXN0dXJhZGFzIG5vIG1lc21vIG7Dum1lcm8uIiwKICAgICJwZXJfY2xhc3MiOiBbCiAgICAgICJpb3UiLAogICAgICAiZGljZSIsCiAgICAgICJwcmVjaXNpb24iLAogICAgICAicmVjYWxsIgogICAgXSwKICAgICJyZXF1aXJlZCI6IFsKICAgICAgImlvdV9iYWNrZ3JvdW5kIiwKICAgICAgImlvdV9wYXBlciIsCiAgICAgICJpb3VfaGFuZCIsCiAgICAgICJpb3VfaW5rIiwKICAgICAgImRpY2VfcGFwZXIiLAogICAgICAiZGljZV9oYW5kIiwKICAgICAgImRpY2VfaW5rIiwKICAgICAgInByZWNpc2lvbiIsCiAgICAgICJyZWNhbGwiCiAgICBdCiAgfSwKICAibW9kZWwiOiB7CiAgICAiYXJjaGl0ZWN0dXJlIjogIlNlZ0Zvcm1lci1CMCIsCiAgICAiYmFja2JvbmUiOiAibnZpZGlhL21pdC1iMCIsCiAgICAiY2xhc3NlcyI6IHsKICAgICAgIjAiOiAiYmFja2dyb3VuZCIsCiAgICAgICIxIjogInBhcGVyIiwKICAgICAgIjIiOiAiaGFuZCIsCiAgICAgICIzIjogImluayIKICAgIH0sCiAgICAiaW5wdXQiOiB7CiAgICAgICJjaGFubmVscyI6IDMsCiAgICAgICJjb2xvcl9zcGFjZSI6ICJSR0IiLAogICAgICAiaGVpZ2h0IjogNTEyLAogICAgICAibm9ybWFsaXphdGlvbiI6IHsKICAgICAgICAibWVhbiI6IFsKICAgICAgICAgIDAuNDg1LAogICAgICAgICAgMC40NTYsCiAgICAgICAgICAwLjQwNgogICAgICAgIF0sCiAgICAgICAgIm5vdGUiOiAiSWTDqm50aWNhIGFvIGNvbnRyYXRvIGF0dWFsIGRlIGRyYXcvbmV1cmFsLnB5IChfTUVBTi9fU1REKS4gTyBPTk5YIG5vdm8gbWFudMOpbSBpbnB1dCAnaW1hZ2UnIDUxMng1MTIgbm9ybWFsaXphZG8gYXNzaW0sIHBhcmEgYSBpbnRlZ3Jhw6fDo28gbsOjbyBtdWRhciBvIHByw6ktcHJvY2Vzc2FtZW50by4iLAogICAgICAgICJzdGQiOiBbCiAgICAgICAgICAwLjIyOSwKICAgICAgICAgIDAuMjI0LAogICAgICAgICAgMC4yMjUKICAgICAgICBdCiAgICAgIH0sCiAgICAgICJ3aWR0aCI6IDUxMgogICAgfSwKICAgICJudW1fY2xhc3NlcyI6IDQsCiAgICAib3V0cHV0IjogewogICAgICAibmFtZSI6ICJsb2dpdHMiLAogICAgICAibm90ZSI6ICJMb2dpdHMgcG9yIGNsYXNzZSwgc2VtIHNvZnRtYXggZW1idXRpZG8uIEEgaW50ZWdyYcOnw6NvIChDb2RleCwgZm9yYSBkZXN0ZSBlc2NvcG8pIGFwbGljYSBhcmdtYXg6IHBhcGVyX21hc2sgPSAoYXJnbWF4ID09IDEpLCBoYW5kX21hc2sgPSAoYXJnbWF4ID09IDIpLCBpbmtfbWFzayA9IChhcmdtYXggPT0gMykuIHBhcGVyX3ByZXNlbmNlIMOpIERFUklWQURPIGRhIG3DoXNjYXJhIChmcmHDp8OjbyBkZSBwaXhlbHMgcGFwZXIgYWNpbWEgZG8gbGltaWFyIGRlIGludGVncmHDp8OjbyksIG7Do28gdW1hIGNhYmXDp2EgYXV4aWxpYXI6IGRlY2lzw6NvIGRvY3VtZW50YWRhLCBtb2RlbG8gZGUgY2FiZcOnYSDDum5pY2EsIHNlbSBzaW1wbGlmaWNhw6fDo28gZGEgc2VnbWVudGHDp8OjbyBwYXJhIGNhYmVyIG5vIGNvbnRyYXRvIGFudGlnbyBkZSAyIGNhbmFpcy4iLAogICAgICAic2hhcGUiOiBbCiAgICAgICAgMSwKICAgICAgICA0LAogICAgICAgIDUxMiwKICAgICAgICA1MTIKICAgICAgXQogICAgfSwKICAgICJwcmV0cmFpbmVkIjogdHJ1ZSwKICAgICJwcmV0cmFpbmVkX3NvdXJjZSI6ICJIdWdnaW5nIEZhY2UgdHJhbnNmb3JtZXJzIChTZWdmb3JtZXJGb3JTZW1hbnRpY1NlZ21lbnRhdGlvbiwgaWQgbnZpZGlhL21pdC1iMCkuIFRyYW5zZmVyIGxlYXJuaW5nOiBlbmNvZGVyIHByw6ktdHJlaW5hZG8gbm8gSW1hZ2VOZXQsIGNhYmXDp2EgZGUgc2VnbWVudGHDp8OjbyBub3ZhIGNvbSA0IGNsYXNzZXMuIgogIH0sCiAgInByb2plY3QiOiAiZHJhdy1zZWdtZW50YXRpb24tdjEiLAogICJ0cmFpbmluZyI6IHsKICAgICJiYXRjaF9zaXplIjogOCwKICAgICJjaGVja3BvaW50IjogImNoZWNrcG9pbnRzL2Jlc3QucHQgKG1lbGhvciB2YWxfbWVhbl9pb3U7IGNoZWNrcG9pbnRzL2xhc3QucHQgYSBjYWRhIMOpcG9jYSkiLAogICAgImVhcmx5X3N0b3BwaW5nIjogewogICAgICAibW9kZSI6ICJtYXgiLAogICAgICAibW9uaXRvciI6ICJ2YWxfbWVhbl9pb3UiLAogICAgICAicGF0aWVuY2VfZXBvY2hzIjogNQogICAgfSwKICAgICJlcG9jaHMiOiAzMCwKICAgICJncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHMiOiAxLAogICAgImxlYXJuaW5nX3JhdGUiOiA2ZS0wNSwKICAgICJtaXhlZF9wcmVjaXNpb24iOiAiYmYxNiAoZnAxNiBzZSBhIEdQVSBuw6NvIHN1cG9ydGFyIGJmMTYpIiwKICAgICJub3RlIjogIkJhdGNoIDggZW0gNTEyeDUxMiBjYWJlIG51bWEgVDQvUDEwMCBkZSAxNiBHQiBjb20gU2VnRm9ybWVyLUIwLiBTZSBob3V2ZXIgT09NLCBiYWl4YXIgcGFyYSBiYXRjaCA0IGNvbSBncmFkaWVudF9hY2N1bXVsYXRpb25fc3RlcHM9MiAoYmF0Y2ggZWZldGl2byBpZ3VhbCkgZW0gdmV6IGRlIHJlZHV6aXIgYSByZXNvbHXDp8OjbzogdHJhw6dvIGZpbm8gZGVwZW5kZSBkYSByZXNvbHXDp8Ojby4iLAogICAgIm51bV93b3JrZXJzIjogNCwKICAgICJvcHRpbWl6ZXIiOiAiQWRhbVciLAogICAgInNjaGVkdWxlciI6ICJjb3NpbmVfd2l0aF93YXJtdXAiLAogICAgInNlZWQiOiA0MiwKICAgICJ3YXJtdXBfcmF0aW8iOiAwLjEsCiAgICAid2VpZ2h0X2RlY2F5IjogMC4wMQogIH0KfQo="
for name, payload in (("train.py", TRAIN_B64), ("config.json", CONFIG_B64)):
    raw = base64.b64decode(payload)
    (BUNDLE / name).write_bytes(raw)
    print(name, len(raw), hashlib.sha256(raw).hexdigest())
print("treinamento congelado no notebook")


## Montagem e preflight dos oito chunks

Os chunks são privados, versão 1: part1–part6 são treino; part7 é validação; part8 é teste. A célula falha se o conteúdo montado não produzir o fingerprint auditado antes de iniciar o treino.


In [ ]:
from pathlib import Path
import hashlib, json, shutil, time

roots = sorted(Path("/kaggle/input/datasets/guzzbr").glob("draw-synthetic-v1-part*/dataset/synthetic"))
if not roots:
    roots = sorted(Path("/kaggle/input").glob("draw-synthetic-v1-part*/dataset/synthetic"))
expected_slugs = {f"draw-synthetic-v1-part{i}" for i in range(1, 9)}
if {p.parents[1].name for p in roots} != expected_slugs or len(roots) != 8:
    raise RuntimeError(f"Inputs sintéticos inesperados: {roots}")
DATA = Path("/kaggle/working/dataset/synthetic")
if DATA.exists(): shutil.rmtree(DATA)
for split in ("train", "val", "test"):
    for kind in ("images", "masks", "ink_full"): (DATA / split / kind).mkdir(parents=True, exist_ok=True)
for part in roots:
    for split in ("train", "val", "test"):
        for kind in ("images", "masks", "ink_full"):
            src_dir = part / split / kind
            if src_dir.is_dir():
                for src in sorted(src_dir.iterdir()):
                    dst = DATA / split / kind / src.name
                    if not dst.exists(): dst.symlink_to(src)
manifest = next((p / "split_manifest.json" for p in roots if (p / "split_manifest.json").is_file()), None)
if manifest is None: raise RuntimeError("split_manifest.json ausente")
shutil.copy2(manifest, DATA / "split_manifest.json")
counts = {s: len(list((DATA / s / "images").glob("*.png"))) for s in ("train", "val", "test")}
if counts != {"train":2400, "val":300, "test":300}: raise RuntimeError(f"Contagens inesperadas: {counts}")

def sha256(path):
    h=hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda:f.read(1024*1024), b""): h.update(block)
    return h.hexdigest()
records=[]
for path in sorted(p for p in DATA.rglob("*") if p.is_file()):
    records.append({"path":path.relative_to(DATA).as_posix(),"bytes":path.stat().st_size,"sha256":sha256(path)})
fp=hashlib.sha256(json.dumps(records,sort_keys=True,separators=(",",":")).encode()).hexdigest()
EXPECTED_FP="4d98f653506439d6cb059be6c1e94883a6f6a394ff916b1c10300f68cbe16dfa"
EXPECTED_MANIFEST="9ba05ab08ad5bf276a96b527777424b8e143958a31d192e7862d458af105c931"
if fp != EXPECTED_FP: raise RuntimeError(f"Dataset SHA fingerprint diverge: {fp}")
if sha256(DATA/"split_manifest.json") != EXPECTED_MANIFEST: raise RuntimeError("Hash do split_manifest diverge")
Path("/kaggle/working/input_hashes.json").write_text(json.dumps({"fingerprint_sha256":fp,"files":records},indent=2))
print("preflight OK", counts, "files",len(records),"fingerprint",fp)

PROVENANCE = json.loads(r'''{"task": "single-frame semantic segmentation of visible ink on paper", "classification": "exploratory synthetic only; no human-reviewed real labels", "datasets": [{"slug": "guzzbr/draw-synthetic-v1-part1", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part2", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part3", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part4", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part5", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part6", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part7", "version": 1, "private": true}, {"slug": "guzzbr/draw-synthetic-v1-part8", "version": 1, "private": true}], "pretrained_cache": {"slug": "guzzbr/mit-b0-cache", "version": 1, "private": true, "model": "nvidia/mit-b0", "snapshot": "80983a413c30d36a39c20203974ae7807835e2b4"}, "excluded_inputs": ["guzzbr/draw-real-capture-pool-v1", "guzzbr/draw-real-seg-pilot-v1", "guzzbr/draw-ink-pairs-v1", "guzzbr/draw-realbg-v2"], "expected_data_fingerprint_sha256": "4d98f653506439d6cb059be6c1e94883a6f6a394ff916b1c10300f68cbe16dfa", "split_manifest_sha256": "9ba05ab08ad5bf276a96b527777424b8e143958a31d192e7862d458af105c931", "data_counts": {"train": 2400, "val": 300, "test": 300}, "baseline": {"name": "synthetic_only", "metrics_sha256": "b7135c498215f2ce1d70b66c8459c78392957347b8800eb4f397bda3433f605d", "split_manifest_sha256": "3a630bf43913bc73397ff2ce3543c57820692c8f0a8adaf3543a9ea803a11304", "val_mean_iou": 0.917703486788386, "val_ink_iou": 0.7433429511659265, "criteria": {"val_mean_iou_min_delta": -0.005, "val_ink_iou_min_delta": -0.01}}}''')
PROVENANCE["code_sha256"] = sha256(BUNDLE/"train.py")
PROVENANCE["config_sha256"] = sha256(BUNDLE/"config.json")
PROVENANCE["input_fingerprint_sha256"] = fp
PROVENANCE["started_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
Path("/kaggle/working/dataset-provenance.json").write_text(json.dumps(PROVENANCE,indent=2))
BASELINE = {"mean_iou":0.917703486788386,"ink":{"iou":0.7433429511659265}}
Path("/kaggle/working/baseline-metrics.json").write_text(json.dumps(BASELINE,indent=2))


## Configuração e contagens

Este kernel treina somente com as máscaras sintéticas verificadas. A validação seleciona o checkpoint; o teste fica fechado até a seleção terminar.


In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/draw")
from training.train import load_config, pair_images_masks, pixel_counts, compute_class_weights
cfg = load_config(BUNDLE / "config.json")
print("modelo", cfg["model"]["architecture"], cfg["model"]["classes"])
for split in ("train", "val", "test"):
    pairs = pair_images_masks(DATA / split)
    counts = pixel_counts(pairs)
    print(split, len(pairs), "pixels/classe", counts.tolist())
    if len(pairs) == 0 or (counts == 0).any(): raise RuntimeError(f"Preflight de rótulos falhou: {split}")


## Treino e gates

O early stopping observa apenas `val_mean_iou`. Após o treino, o script mede o teste uma vez, exporta o ONNX e falha se validação ONNX ou os critérios comparativos da validação não passarem.


In [ ]:
import subprocess, sys
cmd=[sys.executable,str(BUNDLE/"train.py"),"--config",str(BUNDLE/"config.json"),"--data-dir",str(DATA),"--out-dir","/kaggle/working/out","--dataset-provenance-json","/kaggle/working/dataset-provenance.json","--baseline-metrics-json","/kaggle/working/baseline-metrics.json","--require-cuda"]
print("comando"," ".join(cmd))
subprocess.run(cmd,check=True)


## Resultado preliminar

Um score sintético não é evidência de qualidade na câmera. O kernel falha se a comparação com o baseline de validação ou a paridade ONNX não passar. O split de teste é apresentado só depois de concluir a escolha.


In [ ]:
import json
from pathlib import Path
out=Path("/kaggle/working/out")
m=json.load(open(out/"metrics.json"))
print("val mIoU",m["mean_iou"],"test mIoU",m["test"]["mean_iou"])
for label, values in (("val",m),("test",m["test"])):
    print(label)
    for cls in ("background","paper","hand","ink"):
        d=values[cls]; print(cls,{k:d[k] for k in ("iou","dice","precision","recall","support_px")})
print("baseline comparison",m["baseline_comparison"])
print("ONNX validation",m["onnx_validation"])
if m["onnx_validation"].get("status") != "ok": raise RuntimeError("ONNX não validado")
if not m["baseline_comparison"].get("gate_pass"): raise RuntimeError("Candidato não passou critério pré-fixado")
if not m.get("test"): raise RuntimeError("Métricas do teste ausentes")
print("artifact hashes",json.load(open(out/"artifact_hashes.json")))
